<table>
  <tr>
    <td><div align="left"><font size="30">Analytic inverse kinematics</font></div></td>
    <td><img src="https://github.com/petercorke/robotics-toolbox-python/raw/main/docs/notebooks/figs/RobToolBox_RoundLogoB.png?raw=1" width="300"></td>
  </tr>
</table>

<p></p>

(c) Peter Corke 2026

The numerical solvers, `ikine_LM` and friends, start from a guess and iterate.  They find *one* solution, which one depends on the guess, and they can fail even when a solution exists.

For a six-axis arm with a standard Denavit-Hartenberg model there is another way.  The equations of the inverse kinematics are **derived once, symbolically**, and then solved directly for any pose: no guess, no iteration, *all* of the solutions, and a clear answer when the pose cannot be reached.  The derivation takes a few seconds.  The solution can also be written out as source code in Python, C or C++ -- this code is fast, efficient and suitable for embedded control with zero dependency on the Toolbox or SymPy.

The generated code is released under the MIT licence, the same as the Toolbox, and every generated file says so in its header.

Two methods are used, and the Toolbox chooses between them from the structure of the arm:

| Arm | Method |
|---|---|
| three consecutive joint axes meet at a point (a spherical wrist), or are parallel | closed form |
| neither, for example the Kinova Jaco | general 6R, Raghavan and Roth [[1]](#references) |

This needs SymPy, which is not installed with the Toolbox by default: `pip install roboticstoolbox-python[ik]`.  The extra also installs `python-flint`, which makes the derivation much faster.

In [ ]:
# RTB_BOOTSTRAP_CELL: installs the Toolbox (Colab, JupyterLite or local), generated by sync_bootstrap.py from _rtb_nb_bootstrap.py, do not edit

import subprocess
import sys

async def ensure_installed(extras: tuple[str, ...] | list[str]=()) -> bool:
    if sys.platform == 'emscripten':
        try:
            import piplite
            install = piplite.install
        except ImportError:
            import micropip
            install = micropip.install
        await install('spatialmath-python', deps=False)
        await install('ipympl')
        if 'ik' in extras:
            await install('sympy')
        await install('roboticstoolbox-python')
        where, colab = ('in browser', False)
    else:
        try:
            import google.colab
        except ImportError:
            where, colab = ('locally', False)
            if 'ik' in extras:
                try:
                    import sympy
                except ImportError:
                    print("This notebook needs SymPy: pip install 'roboticstoolbox-python[ik]'")
        else:
            print('Installing roboticstoolbox-python...')
            requirement = 'roboticstoolbox-python'
            if extras:
                requirement += '[' + ','.join(extras) + ']'
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', requirement], check=True)
            where, colab = ('on Colab', True)
    import roboticstoolbox
    version = getattr(roboticstoolbox, '__version__', 'unknown')
    print(f'Running {where} using RTB v{version}')
    return colab

COLAB = await ensure_installed(extras=['ik'])


In [ ]:
import numpy as np
import roboticstoolbox as rtb
from spatialmath import SE3
import matplotlib.pyplot as plt

%matplotlib ipympl
plt.close("all")

np.set_printoptions(precision=4, suppress=True, linewidth=120)

## Contents
- [A robot with a spherical wrist: the Puma 560](#A-robot-with-a-spherical-wrist:-the-Puma-560)
- [A robot with a prismatic joint: the Stanford arm](#A-robot-with-a-prismatic-joint:-the-Stanford-arm)
- [The general case: the Kinova Jaco](#The-general-case:-the-Kinova-Jaco)
- [Trajectories](#Trajectories)
- [Generating code](#Generating-code)
- [What it can and can't do](#What-it-can-and-can't-do)
- [References](#References)

## A robot with a spherical wrist: the Puma 560

We start with a classic, defined by standard DH parameters.

In [ ]:
puma = rtb.models.DH.Puma560()
print(puma)

`analytic_ik()` derives the equations for this robot and returns a solver.  The first call takes a second or so, and the solver is kept, so later calls for the same robot are instant.

In [ ]:
ik = puma.analytic_ik()
print(ik)

The three wrist axes meet at a point, so this is solved in closed form as indicated by the `print` above.


In [ ]:
puma.isspherical()


To pose the problem we need a pose that we know can be reached, so we take the forward kinematics of a joint configuration that is within the joint limits.

In [ ]:
q = np.array(np.radians([10, 50, -45, 20, -30, 25])) # joint angles given in degrees
T = puma.fkine(q)
T

`solve` gives **all** the solutions for this pose.  The result is an `IKSolution`, the same class that the numerical solvers return, and its rows are the alternatives.

In [ ]:
sol = ik.solve(T)
print(sol.success, len(sol))
with np.printoptions(precision=2):
    print(np.degrees(sol.q))

Each row is a different arm configuration, for example elbow up or down, wrist flipped or not, and every one reaches the pose.  The configuration we started from is among them.

In [ ]:
errors = [np.abs(puma.fkine(row) - T).max() for row in sol]
print("largest error in the pose:", max(errors))
print("includes the original configuration:", any(np.allclose(row, q) for row in sol))

By default solutions that violate the joint limits are discarded, as the numerical solvers do.  Revolute angles are first shifted by whole turns, since $q$ and $q \pm 2\pi$ are the same pose, so a solution is only rejected if no equivalent angle is within the limits.  With `joint_limits=False` we get everything the equations allow.

In [ ]:
free = ik.solve(T, joint_limits=False)
print(len(sol), "solutions within the limits, of", len(free))

### One solution, as the numerical solvers give

The `ikine_` methods of a robot return one solution, and `ikine_analytic` is the analytic version.  It returns the solution nearest to `q0`, so it can be used in place of `ikine_LM`.

In [ ]:
q0 = q + 0.3
sol = puma.ikine_analytic(T, q0=q0)
print(sol)
print(np.degrees(sol.q))

`iterations` and `searches` are zero because there is no search.  The `weights` argument changes what "nearest" means: a weight of zero ignores a joint, and a large weight makes it stay put.  Here the wrist joints are free to be anywhere, and the arm joints stay as near to `q0` as they can.

In [ ]:
sol = puma.ikine_analytic(T, q0=q0, weights=[1, 1, 1, 0, 0, 0])
with np.printoptions(precision=2):
    print(np.degrees(sol.q))

For comparison, the numerical solver, which needs a good guess and finds only one solution.  The time for the analytic solver includes the Python code that checks the limits and picks the nearest solution, and the generated code, below, is much faster:

In [ ]:
%timeit puma.ikine_analytic(T, q0=q0) # analytic solution implemented in Python
%timeit puma.ikine_LM(T, q0=q0) # numerical solution implemented in Python
try:
    %timeit puma.ik_LM(T, q0=q0) # numerical solution implemented in C++
except RuntimeError:
    print("ik_LM needs the compiled extension, so it is not available in the browser")

### A pose that can't be reached

There is no solution if the pose is outside the workspace.  The answer is a failed `IKSolution` that says why, not an exception, and `q` is NaN.

In [ ]:
sol = puma.ikine_analytic(SE3(5, 5, 5))
print(sol)
bool(sol)

## A robot with a prismatic joint: the Stanford arm

The Stanford arm has a sliding joint as its third joint, then a spherical wrist.  Revolute angles are wrapped to within the limits, but the prismatic joint is just a length.

In [ ]:
stanford = rtb.models.DH.Stanford()
print(stanford)

puma.isspherical()

In [ ]:
ik = stanford.analytic_ik()
print(ik)

In [ ]:
q = np.array([0.3, -0.8, 0.5, 0.4, -0.6, 0.9])  # q3 is a distance, in metres
T = stanford.fkine(q)
sol = ik.solve(T)
print(len(sol), "solutions within the joint limits")
with np.printoptions(precision=2):
    print(sol.q)

Joint 3 has the same value in every solution, since the pose fixes how far the arm must extend, and the other joints have the alternatives.

## The general case: the Kinova Jaco

The Jaco has neither a spherical wrist nor three parallel axes, so there is no closed form.  The general 6R method of Raghavan and Roth [[1]](#references) is used instead.  A general six-revolute arm has up to 16 solutions, of which only some are real.

In [ ]:
jaco = rtb.models.DH.Jaco()
print(jaco)

print(jaco.isspherical())

This takes a few seconds, since three different ways of setting up the equations are tried and the best is chosen numerically.

In [ ]:
ik = jaco.analytic_ik()
print(ik)

Note that this time the solution is indicated as being `General6RIK` rather than `ClosedFormIK` as for
the previous examples.

In [ ]:
q = np.array(np.radians([15, -45, 60, 20, -30, 50]))  # joint angles given in degrees
T = jaco.fkine(q)
sol = ik.solve(T, joint_limits=False)
print(len(sol), "solutions")
with np.printoptions(precision=2):
    print(np.degrees(sol.q))

In [ ]:
errors = [np.abs(jaco.fkine(row) - T).max() for row in sol]
print("largest error in the pose:", max(errors))
print("includes the original configuration:", any(np.allclose(row, q) for row in sol))

The method is less well conditioned than the closed form, so it is slower, about a millisecond in Python, and its accuracy falls near singularities.  Every solution is checked against the forward kinematics before it is returned.

`ikine_analytic` works the same way as for the Puma:

In [ ]:
sol = jaco.ikine_analytic(T, q0=q + 0.2)
np.abs(sol.q - q).max()

## Trajectories

A trajectory is an `SE3` with many poses, or an array of shape (N, 4, 4).  The solution has one row of `q` for each pose, as it does for the numerical solvers.

Choosing among alternatives for each pose would let the arm jump between configurations.  So the first pose is solved nearest to `q0`, and each later pose nearest to the solution of the one before.  The path stays on one branch for as long as the arm can.

In [ ]:
qa = np.array([0.2, 0.9, -0.8, 0.3, -0.5, 0.4])
qb = qa + [0.5, -0.3, 0.4, 0.6, -0.2, 0.5]
t = np.linspace(0, 2, 50)  # time, in seconds
Ts = rtb.ctraj(puma.fkine(qa), puma.fkine(qb), t)
len(Ts)

In [ ]:
sol = puma.ikine_analytic(Ts, q0=qa)
print(sol)
sol.q.shape

In [ ]:
rtb.xplot(t, sol.q, wrist=True, labels="q1 q2 q3 q4 q5 q6", block=None)  # arm joints solid, wrist joints dashed

The curves are smooth.  The largest step of any joint between poses is small, and the largest error of the forward kinematics over the whole path is also reported.

In [ ]:
print("largest joint step (deg):", np.degrees(np.abs(np.diff(sol.q, axis=0)).max()))
print("largest pose error:", sol.residual)

If part of the path can't be reached, the poses that can be solved are, while the others are NaN, and `reason` names the first bad pose.

In [ ]:
far = rtb.ctraj(puma.fkine(qa), SE3(1.5, 0, 0.3) * SE3.Rx(np.pi), 20)
sol = puma.ikine_analytic(far, q0=qa)
print(sol.success)
print(sol.reason)
print("poses with a solution:", int((~np.isnan(sol.q).any(axis=1)).sum()), "of", len(far))

## Generating code

The solver can write its solution as source code, for use on a robot controller or in another program.  The resulting code has no dependency on the Toolbox, NumPy or SymPy.

### Closed form

`generate` takes the language, `"python"`, `"c"` or `"c++"`, and the name for the function.  Every file starts with a header that says where it came from, and records the robot's parameters.

In [ ]:
ik = puma.analytic_ik()
source = ik.generate("python", "puma_ik")  # generate Python code to implement Puma 560 IK
print(len(source.splitlines()), "lines")
print("\n".join(source.splitlines()[:28])) # display the first 28 lines of the generated code

The Python code can be used right here.  It returns every solution, and a status that says whether the pose was reached, was out of the workspace, or was singular.

In [ ]:
namespace = {}
exec(source, namespace)
puma_ik = namespace["puma_ik"]

T = puma.fkine([0.2, 0.9, -0.8, 0.3, -0.5, 0.4])
result = puma_ik(T.A)
print(result.status.name, len(result.solutions), "solutions")

The C and C++ code is a single function.  It is plain C99 (or C++17), needs no library, and checks each solution against the forward kinematics.  Here are the first lines of the function and its documentation.

In [ ]:
source = ik.generate("c", "puma_ik")  # generate C code to implement Puma 560 IK
lines = source.splitlines()
print(len(lines), "lines")
start = next(k for k, line in enumerate(lines) if " * T is the 4x4" in line) - 1
print("\n".join(lines[start:start + 40]))  # display the first 40 lines of the generated function

Save it with `open("puma_ik.c", "w").write(source)` and compile it into your own program.  It takes a few microseconds per pose.

The C code is written with embedded use in mind:

- no dynamic memory allocation: no `malloc`, `calloc`, `realloc` or `free`
- no input or output, and no `goto`, `setjmp` or `longjmp`
- no library other than `math.h`, and `string.h` for the general 6R code
- working arrays have a fixed size, and the solutions are written to an array that the caller provides
- no recursion and no variable-length arrays

These are the habits that coding standards such as MISRA C, the JPL C standard and NASA's "Power of 10" ask for, but the code has **not** been checked against any of them.  It does compile without warnings as C99 with `-pedantic -Wall -Wextra -Wvla`.  The C++ code is different: it returns a `std::vector`, which allocates.

### Rust

The closed form solution can also be written in Rust.  It is a single function, with no crates other than the standard library, which returns the solutions and a status.  This code has been compiled and run, and gives the same solutions as the Python.  It collects the solutions in a `Vec`, so unlike the C it allocates, and the general 6R method isn't available in Rust yet.

In [ ]:
source = ik.generate("rust", "puma_ik")  # generate Rust code to implement Puma 560 IK
lines = source.splitlines()
print(len(lines), "lines")
start = next(k for k, line in enumerate(lines) if "`t` is the 4x4" in line)
print("\n".join(lines[start:start + 30]))  # display the first 40 lines of the generated function

### General 6R

The same call works for the Jaco.  The code is longer, since it must find the roots of a polynomial matrix, so it includes a small linear algebra kernel: LU, QR, SVD and an eigenvalue solver, in plain C, so that it too needs no library.

The linear algebra kernel is about 300 lines of plain C99, written for this project from the standard algorithms (LU, Householder QR, one-sided Jacobi SVD, and the Francis QR iteration for eigenvalues; the references are in the source).  It is covered by the same MIT licence, and is tested against NumPy, but it has had far less use than LAPACK.

In [ ]:
gen = jaco.analytic_ik()
source = gen.generate("c", "jaco_ik")
print(len(source.splitlines()), "lines")
print("\n".join(source.splitlines()[:12]))

In [ ]:
source = gen.generate("python", "jaco_ik")
namespace = {}
exec(source, namespace)
result = namespace["jaco_ik"](jaco.fkine([0.3, -0.8, 1.1, 0.4, -0.6, 0.9]).A)
print(result.status.name, len(result.solutions), "solutions")

Currently, the code generation for the general IK case is only supported for

| Language | Dependencies |
|---|---|
| Python | NumPy, and SciPy if installed |
| C | none beyond `math.h` and `string.h`: it has its own small linear-algebra kernel |
| C++ | none beyond the C++17 standard library, and not Eigen: it is the same C kernel with a thin C++ wrapper |

`generate("rust")` for the Jaco raises `NotImplementedError`, the general method would need the linear algebra kernel ported to Rust, or call out to a linear-algebra crate such as `nalgebra`. This is future work.

## What it can and can't do

- six joints, standard DH parameters (modified DH is not supported)
- the solution is derived for the arm as it is, the base and tool transforms are handled for you
- an arm with a spherical wrist or three parallel axes is solved in closed form, any other six-revolute arm by the general method
- code generation: Python, C and C++ for both methods, Rust for the closed form only
- an arm with a **prismatic joint** and no such structure is not supported, and `analytic_ik()` says so (`NotImplementedError`)
- six joints only: no redundant arms (seven or more joints) and no shorter ones

For a robot this doesn't handle, `ikine_LM`, `ikine_NR`, `ikine_GN` or `ikine_QP` still work.

## References

<a id="references"></a>

1. M. Raghavan and B. Roth, "Inverse kinematics of the general 6R manipulator and related linkages," *Journal of Mechanical Design*, vol. 115, no. 3, pp. 502–508, September 1993.